In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Dataset_Day6.csv")

# Fill numerical missing values with median
df['Bathroom'] = df['Bathroom'].fillna(df['Bathroom'].median())
df['Parking']  = df['Parking'].fillna(df['Parking'].median())

# Fill categorical missing values with mode
df['Furnishing'] = df['Furnishing'].fillna(df['Furnishing'].mode()[0])
df['Type']       = df['Type'].fillna(df['Type'].mode()[0])

print("Missing values after treatment:")
print(df.isnull().sum())

# Outlier detection and removal using IQR
original_len = len(df)
outlier_idx  = set()

for col in ['Area', 'BHK', 'Bathroom', 'Parking', 'Price']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    mask = (df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)
    outlier_idx.update(df[mask].index.tolist())

pct = len(outlier_idx) / original_len * 100
print(f"Outlier rows: {len(outlier_idx)} ({pct:.1f}%)")

if pct > 30:
    print("Reduction > 30% — removing outliers for Price only")
    Q1, Q3 = df['Price'].quantile(0.25), df['Price'].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df['Price'] >= Q1 - 1.5*IQR) & (df['Price'] <= Q3 + 1.5*IQR)]
else:
    print("Reduction < 30% — removing all outlier rows")
    df = df.drop(index=outlier_idx)

df = df.reset_index(drop=True)
print(f"Rows remaining: {len(df)}")

Missing values after treatment:
Area           0
BHK            0
Bathroom       0
Furnishing     0
Parking        0
Price          0
Status         0
Transaction    0
Type           0
dtype: int64
Outlier rows: 179 (14.2%)
Reduction < 30% — removing all outlier rows
Rows remaining: 1080


In [2]:
cat_cols = ['Furnishing', 'Status', 'Transaction', 'Type']
df = pd.get_dummies(df, columns=cat_cols, drop_first=False)

# Convert True/False to 1/0
df = df.apply(lambda x: x.astype(int) if x.dtype == bool else x)

print("Columns after OHE:", list(df.columns))
print("Shape:", df.shape)

Columns after OHE: ['Area', 'BHK', 'Bathroom', 'Parking', 'Price', 'Furnishing_Furnished', 'Furnishing_Semi-Furnished', 'Furnishing_Unfurnished', 'Status_Almost_ready', 'Status_Ready_to_move', 'Transaction_New_Property', 'Transaction_Resale', 'Type_Apartment', 'Type_Builder_Floor']
Shape: (1080, 14)


In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

X = df.drop('Price', axis=1)
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50)

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
y_pred_lr = model_lr.predict(X_test)

n, k = X_test.shape
r2_lr     = r2_score(y_test, y_pred_lr)
adj_r2_lr = 1 - (1 - r2_lr) * (n - 1) / (n - k - 1)
mae_lr    = mean_absolute_error(y_test, y_pred_lr)

print("=== Multiple Linear Regression ===")
print(f"R²          : {r2_lr:.4f}")
print(f"Adjusted R² : {adj_r2_lr:.4f}")
print(f"MAE         : {mae_lr:,.2f}")

=== Multiple Linear Regression ===
R²          : 0.6922
Adjusted R² : 0.6724
MAE         : 4,989,428.49


In [4]:
from sklearn.linear_model import Ridge

model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)
y_pred_ridge = model_ridge.predict(X_test)

r2_ridge     = r2_score(y_test, y_pred_ridge)
adj_r2_ridge = 1 - (1 - r2_ridge) * (n - 1) / (n - k - 1)
mae_ridge    = mean_absolute_error(y_test, y_pred_ridge)

print("=== Ridge Regression ===")
print(f"R²          : {r2_ridge:.4f}")
print(f"Adjusted R² : {adj_r2_ridge:.4f}")
print(f"MAE         : {mae_ridge:,.2f}")

=== Ridge Regression ===
R²          : 0.6923
Adjusted R² : 0.6725
MAE         : 4,989,205.94


In [5]:
from sklearn.linear_model import Lasso

model_lasso = Lasso(alpha=1000)
model_lasso.fit(X_train, y_train)
y_pred_lasso = model_lasso.predict(X_test)

r2_lasso     = r2_score(y_test, y_pred_lasso)
adj_r2_lasso = 1 - (1 - r2_lasso) * (n - 1) / (n - k - 1)
mae_lasso    = mean_absolute_error(y_test, y_pred_lasso)

print("=== Lasso Regression ===")
print(f"R²          : {r2_lasso:.4f}")
print(f"Adjusted R² : {adj_r2_lasso:.4f}")
print(f"MAE         : {mae_lasso:,.2f}")

=== Lasso Regression ===
R²          : 0.6923
Adjusted R² : 0.6725
MAE         : 4,989,249.22


In [ ]:
#Lasso Regression IMPROVED the model compared to Linear Regression.
#Lasso and Ridge performed EQUALLY.